# JEDI-7B grounding smoke test

Goal: prove JEDI-7B loads, responds to a click query on a DesignBench screenshot, and returns coords that rescale to the right spot on the original image.

**Prereqs:** Colab Pro (A100 priority), HF read token, Google Colab VS Code extension.

Run cells top-to-bottom. If cell 6's red dot lands on the target element, JEDI is go for Day 1. See cell 7 for what worked, what fails silently, and how to reuse.

## Cell 1 — Bootstrap: Drive mount, repo clone, deps

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/isaacau502/GUI-grounded-gen'
REPO_DIR = '/content/GUI-grounded-gen'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'vllm', 'qwen-vl-utils', 'huggingface_hub', 'hf_transfer',
                'matplotlib', 'pillow'], check=True)

print('Bootstrap complete.')

## Cell 2 — HF token

VS Code Colab kernel can't pop the permission dialog that `userdata.get()` needs. Prompt via `getpass` instead — paste your HF read token when it asks. Runs once per session.

In [ ]:
import os, getpass
from huggingface_hub import HfApi

# VS Code Colab kernel can't access userdata.get() (no UI for permission prompt).
# Use getpass — prompt stays in session env, never hits notebook outputs.
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')

me = HfApi().whoami()
print(f"HF authed as: {me.get('name', '?')}")

## Cell 3 — Weight download (JEDI + OmniParser fallback)

Two-step for speed: download to `/content/` (local SSD, ~100 MB/s from HF) then copy to Drive (persists across disconnects). `hf_transfer` enabled for xet fast path. Idempotent — re-run skips if already in Drive.

In [ ]:
import os, shutil
from huggingface_hub import snapshot_download

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

JEDI_LOCAL = '/content/jedi-weights'
JEDI_DRIVE = '/content/drive/MyDrive/jedi-weights'
OMNI_LOCAL = '/content/omniparser-weights'
OMNI_DRIVE = '/content/drive/MyDrive/omniparser-weights'

def fetch(repo_id, local_dir, drive_dir, sentinel='config.json'):
    if os.path.exists(os.path.join(drive_dir, sentinel)):
        print(f'{repo_id}: already at {drive_dir}, skipping.')
        return
    print(f'{repo_id}: downloading to {local_dir}...')
    snapshot_download(repo_id=repo_id, local_dir=local_dir,
                      resume_download=True, token=os.environ['HF_TOKEN'])
    print(f'{repo_id}: copying to {drive_dir}...')
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
    print(f'{repo_id}: done.')

fetch('xlangai/Jedi-7B-1080p', JEDI_LOCAL, JEDI_DRIVE)

try:
    fetch('microsoft/OmniParser-v2.0', OMNI_LOCAL, OMNI_DRIVE)
except Exception as e:
    print(f'OmniParser fetch failed (fallback only, non-blocking): {e}')

## Cell 4 — Load JEDI via vLLM

Detect GPU, pick dtype (bf16 on A100/H100, fp16 elsewhere). Colab Pro gives A100 priority but not guarantee.

In [ ]:
import torch
from vllm import LLM, SamplingParams

gpu = torch.cuda.get_device_name(0)
dtype = 'bfloat16' if ('A100' in gpu or 'H100' in gpu) else 'float16'
print(f'GPU: {gpu}')
print(f'dtype: {dtype}')

llm = LLM(
    model='/content/drive/MyDrive/jedi-weights',
    dtype=dtype,
    gpu_memory_utilization=0.9,
    max_model_len=8192,
    limit_mm_per_prompt={'image': 1},
)
print('JEDI loaded.')

## Cell 5 — Smoke query

Set `SCREENSHOT_PATH` and `TARGET_DESC`, run.

**Working prompt pattern:** `"Click <TARGET_DESC>. Output only: pyautogui.click(x=<int>, y=<int>)"`

JEDI needs a concrete element name. Vague targets like "the primary CTA button" trigger chat mode ("The image does not contain any call-to-action buttons...") instead of grounding. Always name the element by its visible label (e.g., `"the 'Upload Photo' button"`).

Image preprocessed with `smart_resize` (factor=28, Qwen2.5-VL patch size) and passed as a base64 data URL via vLLM's OpenAI-compatible `llm.chat()` API.

In [ ]:
import io, base64
from PIL import Image
from qwen_vl_utils import smart_resize

SCREENSHOT_PATH = '/content/drive/MyDrive/samples/1.png'
TARGET_DESC = "the 'Upload Photo' button"

img = Image.open(SCREENSHOT_PATH).convert('RGB')
orig_w, orig_h = img.size
resized_h, resized_w = smart_resize(orig_h, orig_w, factor=28,
                                    min_pixels=256*28*28, max_pixels=1280*28*28)
img_resized = img.resize((resized_w, resized_h))
print(f'Orig: {orig_w}x{orig_h}  Resized: {resized_w}x{resized_h}')

buf = io.BytesIO(); img_resized.save(buf, format='PNG')
data_url = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"

prompt = f"Click {TARGET_DESC}. Output only: pyautogui.click(x=<int>, y=<int>)"
messages = [{'role': 'user', 'content': [
    {'type': 'image_url', 'image_url': {'url': data_url}},
    {'type': 'text', 'text': prompt},
]}]
outputs = llm.chat(messages, sampling_params=SamplingParams(temperature=0.0, max_tokens=64))
raw = outputs[0].outputs[0].text
print(f'Raw JEDI output: {raw}')

## Cell 6 — Parse coord, rescale to orig pixels, visualize

JEDI returns coords in the smart_resize'd image space. Inverse rescale to original pixel coords, draw red dot on the original image, save to Drive.

**Output format JEDI returns:** `x=<float> y=<float>` (in resized space).

**Rescale:** `orig_x = cx_resized * orig_w / resized_w` (same for y).

In [ ]:
import os, re
from PIL import ImageDraw

patterns = [
    r'x\s*=\s*([\d.]+)\s*,?\s*y\s*=\s*([\d.]+)',
    r'pyautogui\.click\(\s*x\s*=\s*([\d.]+)\s*,\s*y\s*=\s*([\d.]+)',
    r'\[(\d+)\s*,\s*(\d+)\]',
    r'\((\d+)\s*,\s*(\d+)\)',
]
match = None
for p in patterns:
    match = re.search(p, raw)
    if match: break
assert match, f'No coord in: {raw}'

cx_r, cy_r = float(match.group(1)), float(match.group(2))
click_x = int(cx_r * orig_w / resized_w)
click_y = int(cy_r * orig_h / resized_h)
print(f'Click resized ({cx_r}, {cy_r}) -> orig ({click_x}, {click_y})')

viz = img.copy()
draw = ImageDraw.Draw(viz)
r = 14
draw.ellipse((click_x - r, click_y - r, click_x + r, click_y + r),
             fill='red', outline='white', width=3)

out_dir = '/content/drive/MyDrive/jedi-smoke-out'
os.makedirs(out_dir, exist_ok=True)
out_path = f'{out_dir}/smoke.png'
viz.save(out_path)
print(f'Saved: {out_path}')
viz

## What worked / how to use

### Config that works
- **Runtime:** Colab Pro A100-SXM4-40GB, vLLM bf16
- **Model:** `xlangai/Jedi-7B-1080p`, weights cached at `/content/drive/MyDrive/jedi-weights/`
- **Prompt pattern:** `"Click <TARGET_DESC>. Output only: pyautogui.click(x=<int>, y=<int>)"`
- **Output format JEDI returns:** `x=<float> y=<float>` (resized-image space)
- **Coord rescale:** `orig_x = cx_resized * orig_w / resized_w`
- **Image preprocess:** `smart_resize(h, w, factor=28, min_pixels=256*28*28, max_pixels=1280*28*28)`, then base64 data URL via `llm.chat` with `image_url`

### What fails silently
- **Vague targets** ("the primary CTA button") → JEDI falls back to chat mode, replies in natural language. Always name element by visible label.
- **`userdata.get('HF_TOKEN')`** in VS Code Colab kernel → timeout. Use `getpass` (cell 2).
- **`{'type': 'image', 'image': PIL.Image}`** in vLLM chat API → `NotImplementedError: Unknown part type: image`. Must use `image_url` with base64 data URL.
- **`smart_resize(h, w, min_pixels=..., max_pixels=...)`** without `factor=28` → `TypeError: missing required positional argument 'factor'`.

### How to run
1. Open notebook in VS Code with the **Google Colab** extension, connect to A100 runtime
2. Run cells 1–4 in order: bootstrap → HF token → weight download → JEDI load. After the first run, cell 3 skips (weights already in Drive) and cell 4 takes ~30s.
3. In cell 5: set `SCREENSHOT_PATH` (to a `.png` in Drive) and `TARGET_DESC` (concrete element label). Run.
4. Cell 6 parses the coord, rescales to original pixel space, draws a red dot, and saves to `/content/drive/MyDrive/jedi-smoke-out/smoke.png`.

### Day 1 handoff
Lift cells 5–6 into `grounding/jedi.py`:
- Wrap in a function `ground(screenshot_path, target_desc) -> (x, y)`
- Loop over the 111 DesignBench samples driven by oracle issue labels
- Cache `{sample_id, target_desc, click_x, click_y, raw}` as JSON in Drive
- Download the JSON cache to the Mac for the repair pipeline